# Oct 2025: New sensing model with "fractionated" Mormyromasts, Amp/Kollen same as before

In [ ]:
import sys; sys.path.insert(0, 'calibration')  # utils_calibrate* live in calibration/

In [ ]:
import utils_calibrate as uc
import numpy as np
import matplotlib.pyplot as plt
from electric import fish_forward
from cfg import ELECTRIC_CONSTANTS, AGENT_PARAMS, ENV_PARAMS
from MAEFish import EFishAgent
import pandas as pd
import math


## Load CFG constants

In [ ]:
# Constants from your code
k_coulomb = ELECTRIC_CONSTANTS["k_coulomb"]  # Coulomb's constant
epsilon_0 = ELECTRIC_CONSTANTS["epsilon_0"]  # Vacuum permittivity (F/m)
epsilon0 = epsilon_0
chi = ELECTRIC_CONSTANTS["food_contrast"]
assert ELECTRIC_CONSTANTS["food_contrast"] == ELECTRIC_CONSTANTS["fish_contrast"]


mVcm_to_Vm = ELECTRIC_CONSTANTS["mVcm_to_Vm"]
print(f"mV/cm to V/m conversion factor: {mVcm_to_Vm:.2f}")
Vm_to_mVcm =ELECTRIC_CONSTANTS["Vm_to_mVcm"]
print(f"V/m to mV/cm conversion factor: {Vm_to_mVcm:.2f}")
cm_to_m = 0.01

fish_radius_m = AGENT_PARAMS["body_radius"] * cm_to_m
fish_diameter_m = 2 * fish_radius_m 

# Misc
# needed for monopole charge calculation
fish_monopole_distance = AGENT_PARAMS["monopole_positions_ego"]
fish_monopole_distance = (
    np.linalg.norm(fish_monopole_distance[0] - fish_monopole_distance[1]) * cm_to_m
)  # meters
print(f"fish_monopole_distance: {fish_monopole_distance:.2e} m")

# Sensor thresholds: V/m (correct in cfg.py)
ampullary_sensor_min = AGENT_PARAMS["ampullary_sensor_min"]
ampullary_sensor_max = AGENT_PARAMS["ampullary_sensor_max"]
mormyromast_sensor_min = AGENT_PARAMS["mormyromast_sensor_min"]
mormyromast_sensor_max = AGENT_PARAMS["mormyromast_sensor_max"]
knollen_sensor_min = AGENT_PARAMS["knollen_sensor_min"] # No max for Knollen

print("\nSensor thresholds (V/m):")
print(f"Ampullary: {ampullary_sensor_min:.2e} -- {ampullary_sensor_max:.1e} V/m ({ampullary_sensor_min*Vm_to_mVcm:.1e} -- {ampullary_sensor_max*Vm_to_mVcm:.1e} mV/cm)")
print(f"Mormyromast: {mormyromast_sensor_min:.3f} -- {mormyromast_sensor_max:.3f} V/m ({mormyromast_sensor_min*Vm_to_mVcm:.3f} -- {mormyromast_sensor_max*Vm_to_mVcm:.3f} mV/cm)")
print(f"Knollen: {knollen_sensor_min:.3e} V/m ({knollen_sensor_min*Vm_to_mVcm:.3e} mV/cm)")

# Detection ranges: m
amp_food_detection_range = AGENT_PARAMS["amp_food_detection_range"]
amp_fish_detection_range = AGENT_PARAMS["amp_fish_detection_range"]
morm_food_detection_range = AGENT_PARAMS["morm_food_detection_range"]
morm_agent_detection_range = AGENT_PARAMS["morm_agent_detection_range"]
knollen_agent_detection_range = AGENT_PARAMS["knollen_agent_detection_range"]

print("\nDetection ranges (m):")
print(f"Ampullary food detection range: {amp_food_detection_range:.2f} m")
print(f"Ampullary fish detection range: {amp_fish_detection_range:.2f} m")
print(f"Mormyromast food detection range: {morm_food_detection_range:.2f} m")
print(f"Mormyromast agent detection range: {morm_agent_detection_range:.2f} m")
print(f"Knollen agent detection range: {knollen_agent_detection_range:.2f} m")

food_intrinsic_dipole_moment = ENV_PARAMS["food_intrinsic_dipole_moment"][0]
print(f"\nFood intrinsic dipole moment: {food_intrinsic_dipole_moment} Cm")
fish_intrinsic_dipole_moment = AGENT_PARAMS["fish_intrinsic_dipole_moment"][0]
print(f"Fish intrinsic dipole moment: {fish_intrinsic_dipole_moment} Cm")

## Fish active mono/dipole to match literature
Given the axial field estimates from literature, what is the fish mono/dipole that produces it?


Chen 2005 -- 
"The lateral field strength measured just outside the skin ranged from 0.63 mV/cm to 1.2 mV/cm with a mean of 0.97±0.23 mV/cm (n=7)."

Therefore, 2 mV/cm at 1cm (because lateral field reported not axial)


In [ ]:
from utils_calibrate_analytic import dipole_moment, monopole_charge, axial_field

# Distances to evaluate (cm → m)
distances_cm = [1, 5, 10, 15, 100]
distances_m = [d * cm_to_m for d in distances_cm]

# Benda 2020, 20  uV/cm at 37.5 cm (from figure)
# E = 20e-6 / 0.01   # 20 microV/cm -> V/m
# r = 0.375          # 37.5 cm -> m
# d = 0.01           # separation 1 cm -> m

# Chen 2005 -- 1mV at 1cm
# "The lateral field strength measured just outside the skin ranged from 0.63 mV/cm to 1.2 mV/cm with a mean of 0.97±0.23 mV/cm (n=7)."
E = 2 * mVcm_to_Vm  # V/m # Note: x2 because lateral field reported not axial
r = 1 * cm_to_m  #  m # literally on the body of the agent
# r = 2 * cm_to_m   # 2 cm -> m # or take 2 cm to account for a 1 cm body radius
d = 1 * cm_to_m  # separation 1 cm -> m


p = dipole_moment(E, r)
q = monopole_charge(p, d)
print("Dipole moment p =", p, "C·m")
print("Monopole charge q =", q, "C")

print("\nAxial field values:")
for r in distances_m:
    E = axial_field(p, r)
    print(f"r = {r*100:.0f} cm -> E = {E:.3e} V/m  ({E*0.01*1e3:.4f} mV/cm)")

fish_eod_dipole_moment = p
fish_eod_monopole_charge = q
print(
    f"\nFish EOD dipole moment set to {fish_eod_dipole_moment:.2e} C·m and monopole charge set to {fish_eod_monopole_charge:.2e} C"
)

## Intrinsic fish charge/moment 

Intrinsic dipoles on agents have been set to be $10^{-6}$ times the active dipoles.

Intrinsic dipoles on food have been set to be $10^{-1}$ times the intrinsic dipoles on fish.


In [ ]:
active_to_intrinsic_ratio = 1e-6
fish_to_food_intrinsic_ratio = 1e-1

fish_intrinsic_dipole_moment = fish_eod_dipole_moment * active_to_intrinsic_ratio
print(f"Fish intrinsic dipole moment set to {fish_intrinsic_dipole_moment:.2e} C·m")

food_intrinsic_dipole_moment = fish_intrinsic_dipole_moment * fish_to_food_intrinsic_ratio
print(f"Food intrinsic dipole moment set to {food_intrinsic_dipole_moment:.2e} C·m")

## Given fish EOD and intrisic moments, what are the dynamic ranges for K, A, M

In [ ]:
# What is the field at knollen sensors due to active fish EOD? 
# Emitting fish at knollen_agent_detection_range
print("\nKNOLLEN: Agent/passive")
for knollen_distance in [knollen_agent_detection_range, knollen_agent_detection_range/10]:
    E_knollen = axial_field(fish_eod_dipole_moment, knollen_distance)
    print(f"Knollen agent distance {knollen_distance:.2f} m, E = {E_knollen*Vm_to_mVcm:.3e} mV/cm")


In [ ]:

print("\nAMPULLARY: Agent/passive")
for agent_distance in [
    amp_fish_detection_range, 
    amp_fish_detection_range/4
    ]:
    E_ampullary = axial_field(fish_intrinsic_dipole_moment, agent_distance)
    print(f"Ampullary agent distance {agent_distance:.3f} m, E = {E_ampullary*Vm_to_mVcm:.3e} mV/cm")


print("\nAMPULLARY: Food/passive")
for food_distance in [amp_food_detection_range, amp_food_detection_range/4]:
    E_ampullary_food = axial_field(food_intrinsic_dipole_moment, food_distance)
    print(f"Ampullary food distance {food_distance:.3f} m, E = {E_ampullary_food*Vm_to_mVcm:.3e} mV/cm")


In [ ]:
# Uses self-EOD → conspecific-as-object image at the receiver's sensor.
# The conspecific is modeled as a conducting sphere with radius ~ body_radius and contrast chi_agent.
from utils_calibrate_analytic import self_image_components
R_agent = AGENT_PARAMS.get("body_radius", 1.0) * cm_to_m  # meters

print("\nMORMYROMAST: Self-EOD baseline")
E_self_EOD = axial_field(fish_eod_dipole_moment, R_agent)
print(f"Agent surface {R_agent:.3f} m, E = {E_self_EOD*Vm_to_mVcm:.3e} mV/cm ({E_self_EOD:.3e} V/m)")
print("No intrinsic charge used in above")

print("\nMORMYROMAST: Agent/active")
chi_agent = ELECTRIC_CONSTANTS.get("agent_contrast", ELECTRIC_CONSTANTS.get("food_contrast", -0.5))
for agent_distance in [morm_agent_detection_range, 
                       morm_agent_detection_range / 2, 
                       morm_agent_detection_range / 4,
                       morm_agent_detection_range / 6,
                       morm_agent_detection_range / 8,
                       morm_agent_detection_range / 10,
                       ]:
    comp = self_image_components(
        q=fish_eod_monopole_charge,
        d=fish_monopole_distance,
        R=R_agent,
        chi=chi_agent,
        x_food=agent_distance,          # treating the other fish as the "object" on-axis
        epsilon0=epsilon_0,
        subtract_baseline=True          # mormyromasts see induced component after CD subtraction
    )
    E_ind = comp["E_induced_at_sensor"]     # equals comp["E_sensed"] when subtract_baseline=True
    print(
        f"Agent distance {agent_distance:.3f} m -> "
        f"E_induced_at_sensor = {E_ind*Vm_to_mVcm:.3e} mV/cm"
    )


print("\nMORMYROMAST: Food/active")
R_food = ENV_PARAMS.get("food_radius", 0.25) * cm_to_m  # meters
chi_food = ELECTRIC_CONSTANTS.get("food_contrast", -0.5)
for food_distance in [morm_food_detection_range, morm_food_detection_range / 2]:
    comp = self_image_components(
        q=fish_eod_monopole_charge,
        d=fish_monopole_distance,
        R=R_food,
        chi=chi_food,
        x_food=food_distance,
        epsilon0=epsilon_0,
        subtract_baseline=True
    )
    E_ind = comp["E_induced_at_sensor"]     # equals comp["E_sensed"] when subtract_baseline=True
    print(
        f"Food distance {food_distance:.3f} m -> "
        f"E_induced_at_sensor = {E_ind*Vm_to_mVcm:.3e} mV/cm"
    )



## Forward: Field calculations assuming all objects placed along one line

## Active Sensing: E_induced as x_food is varied along dipole axis

In [ ]:
from utils_calibrate_analytic import (
    self_image_components,
    cons_image_components,
    self_image_far_field,
    cons_image_far_field,
    compute_induced_to_self_ratio_vs_xvals,
    plot_morm_induced_to_self_ratio_vs_food_dist,
    plot_morm_induced_abs_vs_food_dist,
    plot_morm_sensed_vs_food_dist,
)




x_vals = np.linspace(0.0125, 0.05, 20)  # 1.25 cm (fish + food ~touching) to 5 cm
df = compute_induced_to_self_ratio_vs_xvals(x_vals=x_vals)
print(df.columns)
plot_morm_induced_to_self_ratio_vs_food_dist(df)
plot_morm_induced_abs_vs_food_dist(df)
plot_morm_sensed_vs_food_dist(df)

# Therefore the mormmyromast min-max sensitivity ranges are
E_self_mean = df["E_self_at_sensor (V/m)"].mean()
E_self_std = df["E_self_at_sensor (V/m)"].std()
print(f"E_self_at_sensor mean: {E_self_mean:.2e} V/m, std: {E_self_std:.2e} V/m")

morm_max = E_self_mean * 0.25  # +/- 25%
morm_min = df["E_ind_at_sensor (V/m)"].min()  # Note max at 5 cm!
print(
    f"Mormyromast sensitivity range: {morm_min:.2e} to {morm_max:.2e} V/m [E_self_EOD = {E_self_mean:.2e} V/m]"
)

## Cons-field as x_cons is varied

Note: Discrepancy with calibrate_frac_morm is due to presence of food in the cons-image formula (in below)


In [ ]:
from utils_calibrate_analytic import cons_eod_field_at_sensor, receiver_induced_field

q = fish_eod_monopole_charge
d = fish_diameter_m
x_cons = knollen_agent_detection_range # 1 m

E_cons = cons_eod_field_at_sensor(q, d, x_cons, epsilon0)
E_receiver_induced = receiver_induced_field(
    q, x_cons=1, r=0.01, d=d, chi_r=-0.5, epsilon0=epsilon0
)
print("At Knollenorgan range (1m), Cons EOD field is:", E_cons, "V/m")
print("Induced field at receiver from cons EOD is:", E_receiver_induced, "V/m")
print("Total field at receiver is:", E_cons + E_receiver_induced, "V/m")

In [ ]:
from utils_calibrate_analytic import plot_cons_field_vs_distance

correct_for_receiver_induction = True # Should there be a field induced on receiver?

# Sweep cons distances from 2 cm to 10 cm
x_cons_vals = np.linspace(0.02, 0.12, 50)  # in meters
E_cons_vals = [cons_eod_field_at_sensor(q, d, xc, epsilon0) for xc in x_cons_vals]
if correct_for_receiver_induction:
    E_recv_vals = [
        receiver_induced_field(q, d=d, x_cons=xc, r=d / 2, chi_r=chi, epsilon0=epsilon0)
        for xc in x_cons_vals
    ]
    E_cons_vals = np.array(E_cons_vals) + np.array(E_recv_vals) 

x_cons_baselines = [0.03, 0.05, 0.07, 0.09, 0.10]  # in meters
E_cons_baselines = [
    cons_eod_field_at_sensor(q, d, xc, epsilon0) for xc in x_cons_baselines
]
if correct_for_receiver_induction:
    E_recv_baselines = [
        receiver_induced_field(q, d=d, x_cons=xc, r=d / 2, chi_r=chi, epsilon0=epsilon0)
        for xc in x_cons_baselines
    ]
    E_cons_baselines = np.array(E_cons_baselines) + np.array(E_recv_baselines)

for i, (xc, baseline) in enumerate(zip(x_cons_baselines, E_cons_baselines)):
    print(
        f"x_cons={xc*100:.1f} cm -> E_cons_EOD: {baseline:.2e} V/m, E_cons/E_self: {baseline/E_self_mean}"
    )



# Absolute mode
plot_cons_field_vs_distance(x_cons_vals, E_cons_vals, E_self_mean,
                            plot_relative_to_self=False,
                            plot_cons_spans=True,
                            x_cons_baselines=x_cons_baselines,
                            E_cons_baselines=E_cons_baselines,
                            savepath="morm_cons_field_vs_distance_abs.png"
                            )

# Relative-to-self mode
plot_cons_field_vs_distance(x_cons_vals, E_cons_vals, E_self_mean,
                            plot_relative_to_self=True,
                            plot_cons_spans=True,
                            x_cons_baselines=x_cons_baselines,
                            E_cons_baselines=E_cons_baselines,
                            savepath="morm_cons_field_vs_distance_rel.png"
                            ) 



## Cons-images: Food Induced Fields on top of Cons-EOD
(Just a sanity check, not really needed?)


In [ ]:
if False:
    # Sweep food distances: 1.25 cm to 5 cm
    x_cons = 0.05  # m (fixed cons center distance)
    # x_food_max = x_cons-R-(d/2) # cons-fish and food can at most touch
    x_food_max = x_cons - R - (d / 2)  # cons-fish and food can at most touch
    x_food_min = R + (d / 2)  # self-fish and food can at most touch
    x_foods = np.linspace(x_food_min, x_food_max, 20)

    correct_for_receiver_induction = False  # TODO: Hard to think about this properly

    rows = []
    for x in x_foods:
        exact = cons_image_components(
            q, d, R, chi, x, x_cons, epsilon0, subtract_baseline=False
        )
        # approx = cons_image_far_field(q, d, R, chi, x, x_cons, epsilon0, subtract_baseline=False)
        if correct_for_receiver_induction:
            exact["E_cons_EOD_at_sensor"] += receiver_induced_field(
                q, d=d, x_cons=x_cons, r=d / 2, chi_r=chi, epsilon0=epsilon0
            )
            # approx["E_cons_EOD_at_sensor"] += receiver_induced_field(q, d=d, x_cons=x_cons, r=d/2, chi_r=chi, epsilon0=epsilon0)

        E_cons = np.abs(exact["E_cons_EOD_at_sensor"])
        E_ind = np.abs(exact["E_induced_at_sensor"])
        # E_ind_ff    = np.abs(approx["E_induced_at_sensor"])

        ratio = E_ind / E_cons
        # ratio_ff    = E_ind_ff / np.abs(approx["E_cons_EOD_at_sensor"])

        rows.append(
            {
                "x_food_m": x,
                "x_food_cm": x * 100.0,
                "E_cons_at_sensor (V/m)": E_cons,
                "E_ind_at_sensor (V/m)": E_ind,
                "E_ind/E_cons (exact)": ratio,
                # "E_ind/E_cons (far-field)": ratio_ff,
                "E_sensed (V/m)": exact["E_sensed"],
            }
        )


    df_cons = pd.DataFrame(rows)
    E_cons = df_cons["E_cons_at_sensor (V/m)"].mean()
    print(df_cons["E_cons_at_sensor (V/m)"])

    # 1) Ratio E_ind/E_cons vs x_food
    plt.figure(figsize=(5, 3))
    plt.plot(
        df_cons["x_food_cm"], df_cons["E_ind/E_cons (exact)"], marker="o"
    )
    # plt.plot(df_cons["x_food_cm"], df_cons["E_ind/E_cons (far-field)"], marker='s', label='Far-field')
    plt.yscale("log")
    plt.xlabel("Food distance x_food (cm)")
    plt.ylabel("E_ind / E_cons")
    plt.title("Cons-image: Induced-to-Cons Field Ratio vs Food Distance")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 2) Induced distortion magnitude vs x_food
    plt.figure(figsize=(5, 3))
    plt.plot(
        df_cons["x_food_cm"],
        df_cons["E_ind_at_sensor (V/m)"],
        marker="o",
        label="Exact",
    )
    plt.axhline(E_cons, color="gray", linestyle="--", label="E_cons")
    plt.yscale("log")
    plt.xlabel("Food distance x_food (cm)")
    plt.ylabel("E_ind_at_sensor (V/m)")
    plt.title(f"Cons-image: Induced Field vs Food Distance \nE_cons: {E_cons:.2e} V/m")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 3) Total sensed at sensor (no cons-baseline subtraction) vs x_food
    plt.figure(figsize=(5, 3))
    plt.plot(
        df_cons["x_food_cm"],
        df_cons["E_sensed (V/m)"],
        marker="o",
        label="Exact (E_cons + E_ind)",
    )
    E_cons_mean = df_cons["E_cons_at_sensor (V/m)"].mean()
    plt.axhline(E_cons_mean * 0.75, color="gray", linestyle="--", label="E_cons - 25%")
    plt.axhline(E_cons_mean, color="gray", linestyle="-.", label="E_cons mean")
    plt.axhline(E_cons_mean * 1.25, color="gray", linestyle=":", label="E_cons + 25%")
    plt.yscale("log")
    plt.xlabel("Food distance x_food (cm)")
    plt.ylabel("E_sensed (V/m)")
    plt.title("Cons-image: Sensed Field vs Food Distance")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Summaries
    print("E_ind/E_cons summary:")
    print(df_cons["E_ind/E_cons (exact)"].describe())
    print()
    print("E_ind_at_sensor (V/m) summary:")
    print(df_cons["E_ind_at_sensor (V/m)"].describe())
    print()
    print("E_sensed (V/m) summary:")
    print(df_cons["E_sensed (V/m)"].describe())

## Fractionated random baselines

In [ ]:
from utils_calibrate_analytic import get_frac_rand_grid

num_grid_points = 20
for style in ["linear", 
              "log", 
              "linear_in_distance"
              ]:
    grid = get_frac_rand_grid(style, mag_min=E_self_mean*2e-4, mag_max=E_self_mean, n=num_grid_points, p=3)
    print(f"{style}: {grid}")
    plt.plot(grid, marker="o", label=style, alpha=0.7)

plt.xlabel("Index")
plt.ylabel("|E_cons/E_self|")
plt.title(f"Conspecific Baseline Grid Styles (N={num_grid_points})")
plt.yscale("log")
plt.legend()
plt.savefig("cons_baseline_grid_styles.png")
plt.show()

## Passive reception of EODs by M/A sensors
1. Till what distance from EOD-emitting conspecific can the M/A sensors detect the conspecific-EOD?
2. At what distance from EOD-emitting conspecifc will the M/A sensors saturate due to the conspecific-EOD?


In [ ]:

def compute_range(p_dipole, E, epsilon_0):
    """
    Compute detection range for a dipole field sensed by a threshold-limited sensor.

    Parameters:
        p_dipole (float): Dipole moment (C·m).
        E (float): Field strength (V/m).
        epsilon_0 (float): Vacuum permittivity (F/m).

    Returns:
        float: Detection distance (m).
    """
    return (p_dipole / (2 * np.pi * epsilon_0 * E)) ** (1 / 3)


# Detection/saturation ranges for Ampullary
amp_detect_range = compute_range(
    fish_eod_dipole_moment, ampullary_sensor_min, epsilon_0
)
amp_saturate_range = compute_range(
    fish_eod_dipole_moment, ampullary_sensor_max, epsilon_0
)

print(f"Ampullary detection range: {amp_detect_range:.2f} m")
print(f"Ampullary saturation range: {amp_saturate_range:.2f} m")

# Detection/saturation ranges for Mormyromast
morm_detect_range = compute_range(
    fish_eod_dipole_moment, mormyromast_sensor_min, epsilon_0
)
morm_saturate_range = compute_range(
    fish_eod_dipole_moment, mormyromast_sensor_max, epsilon_0
)

print(f"Mormyromast detection range: {morm_detect_range:.2f} m")
print(f"Mormyromast saturation range: {morm_saturate_range:.2f} m")

# Detection/saturation ranges for Knollen
knollen_detect_range = compute_range(
    fish_eod_dipole_moment, knollen_sensor_min, epsilon_0
)
knollen_saturate_range = compute_range(
    fish_eod_dipole_moment, knollen_sensor_max, epsilon_0
)
print(f"Knollen detection range: {knollen_detect_range:.2f} m")
print(f"Knollen saturation range: {knollen_saturate_range:.2f} m")